<a href="https://colab.research.google.com/github/Glo14/DataEngineeringProjects/blob/main/T%C3%A9cnica_Asistenete_M%26E_Loreto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from caas_jupyter_tools import display_dataframe_to_user

# Cargar datos
file_path = "/content/BBDD_asistente_MyE.xlsx" # Updated file path
df = pd.read_excel(file_path, sheet_name="BBDD")

# 1. Filas con IDs duplicados (todas las filas cuyo ID aparece más de una vez)
dup_rows = df.duplicated(subset=["ID_Participante"], keep=False).sum()

# 2. Registros con Asistencia_% en blanco
asistencia_blank = df["Asistencia_%"].isna().sum()

# 3. Promedio de Asistencia_% para Matriculados = "Sí" en Región = "Sierra"
mask_matric_sierra = (df["Matriculado"].str.strip().str.lower() == "sí") & (df["Región"].str.strip().str.lower() == "sierra")
prom_asistencia_matric_sierra = df.loc[mask_matric_sierra, "Asistencia_%"].mean()

# 4-7. Recuento y % por sexo
total = len(df)
count_by_sex = df["Sexo"].value_counts(dropna=False)
pct_by_sex = (count_by_sex / total) * 100

n_m = int(count_by_sex.get("M", 0))
p_m = float(pct_by_sex.get("M", 0))

n_f = int(count_by_sex.get("F", 0))
p_f = float(pct_by_sex.get("F", 0))

# 8-11. Promedios por sexo
group = df.groupby("Sexo", dropna=False).agg({
    "Prueba_Lectura_0_20":"mean",
    "Prueba_Matemática_0_20":"mean"
})

lectura_m = float(group.loc["M", "Prueba_Lectura_0_20"]) if "M" in group.index else np.nan
matematica_m = float(group.loc["M", "Prueba_Matemática_0_20"]) if "M" in group.index else np.nan
lectura_f = float(group.loc["F", "Prueba_Lectura_0_20"]) if "F" in group.index else np.nan
matematica_f = float(group.loc["F", "Prueba_Matemática_0_20"]) if "F" in group.index else np.nan

# 12-15. Aprobados (>=11) y desaprobados (<11) y %
aprobados_mask = df["Prueba_Matemática_0_20"] >= 11
n_aprob = int(aprobados_mask.sum())
n_desap = int((~aprobados_mask).sum())
pct_aprob = (n_aprob / total) * 100
pct_desap = (n_desap / total) * 100

# 16-18. Mediana de edad por región
mediana_edad_por_region = df.groupby("Región")["Edad"].median()

mediana_edad_costa = float(mediana_edad_por_region.get("Costa", np.nan))
mediana_edad_sierra = float(mediana_edad_por_region.get("Sierra", np.nan))
mediana_edad_selva = float(mediana_edad_por_region.get("Selva", np.nan))

# 19. Correlación lectura vs matemática
corr_lect_mate = df["Prueba_Lectura_0_20"].corr(df["Prueba_Matemática_0_20"])

# 20. Brecha de conocimiento DSR (F - M) en p.p.
def pct_conoce_by_sex(sex):
    sub = df[df["Sexo"] == sex]
    if len(sub) == 0:
        return np.nan
    return 100 * sub["Conoce_DSR_(0=no,1=sí)"].mean()

pct_conoce_f = pct_conoce_by_sex("F")
pct_conoce_m = pct_conoce_by_sex("M")
brecha_pp = pct_conoce_f - pct_conoce_m

# Formatear resultados
def f(x, decimals=2):
    if pd.isna(x):
        return ""
    return round(float(x), decimals)

results = pd.DataFrame({
    "Pregunta":[
        1,2,3,4,5,6,7,8,9,10,11,
        12,13,14,15,16,17,18,19,20
    ],
    "Resultado":[
        dup_rows,
        asistencia_blank,
        f(prom_asistencia_matric_sierra,2),
        f(n_m,0),
        f(p_m,2),
        f(n_f,0),
        f(p_f,2),
        f(lectura_m,2),
        f(matematica_m,2),
        f(lectura_f,2),
        f(matematica_f,2),
        f(n_aprob,0),
        f(pct_aprob,2),
        f(n_desap,0),
        f(pct_desap,2),
        f(mediana_edad_costa,2),
        f(mediana_edad_sierra,2),
        f(mediana_edad_selva,2),
        f(corr_lect_mate,4),
        f(brecha_pp,2)
    ],
    "Unidad/Nota":[
        "filas",
        "registros",
        "% asistencia (promedio)",
        "N° Masculinos",
        "% Masculinos",
        "N° Femeninos",
        "% Femeninos",
        "Promedio Lectura hombres (0-20)",
        "Promedio Matemática hombres (0-20)",
        "Promedio Lectura mujeres (0-20)",
        "Promedio Matemática mujeres (0-20)",
        "N° Aprobados (>=11)",
        "% Aprobados sobre total",
        "N° Desaprobados (<11)",
        "% Desaprobados sobre total",
        "Mediana de edad Costa (años)",
        "Mediana de edad Sierra (años)",
        "Mediana de edad Selva (años)",
        "Coef. correlación (Pearson)",
        "F - M en p.p."
    ]
})

display_dataframe_to_user("Respuestas_EmprendamosYaII", results)

results

In [ ]:
!pip install caas_jupyter_tools

In [ ]:
# --- Librerías ---
import pandas as pd
import matplotlib.pyplot as plt

# --- Cargar el archivo (ajusta la ruta según dónde lo tengas en Colab) ---
df = pd.read_excel("/content/BBDD_asistente_MyE.xlsx", sheet_name="BBDD")

# --- Vista rápida de los primeros registros ---
display(df.head())

# --- 1. Duplicados ---
duplicados = df.duplicated().sum()
print(f"Número de filas duplicadas: {duplicados}")

# --- 2. Valores nulos ---
print("\nValores nulos por columna:")
print(df.isnull().sum())

# --- 3. Estadísticos descriptivos ---
print("\nEstadísticos descriptivos:")
display(df.describe(include="all"))

# --- 4. Correlación entre columnas numéricas ---
print("\nCorrelaciones:")
display(df.corr(numeric_only=True))

# --- 5. Visualización rápida ---
plt.figure(figsize=(8,5))
df.corr(numeric_only=True).style.background_gradient(cmap="Blues")


In [ ]:
from matplotlib import pyplot as plt
_df_0['ID_Participante'].plot(kind='hist', bins=20, title='ID_Participante')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from matplotlib import pyplot as plt
_df_4.plot(kind='scatter', x='ID_Participante', y='Edad', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

In [3]:
# ================================
# Script para resolver las 20 preguntas con la base BBDD_asistente_MyE.xlsx
# ================================

import pandas as pd
import numpy as np

# 1. Cargar la base de datos
df = pd.read_excel("BBDD_asistente_MyE.xlsx")

# Ver columnas para referencia
print("Columnas del dataset:", df.columns.tolist())
print("="*60)

# ===============================
# 1. Filas con IDs duplicados en ID_Participante
# ===============================
duplicados = df[df.duplicated(subset="ID_Participante", keep=False)]
print("1) Filas con IDs duplicados:", len(duplicados))

# ===============================
# 2. Registros con Asistencia_% en blanco
# ===============================
faltantes_asistencia = df["Asistencia_%"].isna().sum()
print("2) Registros con Asistencia_% en blanco:", faltantes_asistencia)

# ===============================
# 3. Promedio Asistencia_% Matriculados=Sí y Región=Sierra
# ===============================
filtro = df[(df["Matriculado"]=="Sí") & (df["Región"]=="Sierra")]
print("3) Promedio Asistencia_% (Matriculados=Sí, Región=Sierra):", filtro["Asistencia_%"].mean())

# ===============================
# 4-7. Recuento y % por sexo
# ===============================
conteo_sexo = df["Sexo"].value_counts()
porc_sexo = df["Sexo"].value_counts(normalize=True)*100
print("4) Recuento masculinos:", conteo_sexo.get("M",0))
print("5) % masculinos:", porc_sexo.get("M",0))
print("6) Recuento femeninos:", conteo_sexo.get("F",0))
print("7) % femeninos:", porc_sexo.get("F",0))

# ===============================
# 8-11. Promedios de lectura y matemática por sexo
# ===============================
promedios = df.groupby("Sexo")[["Prueba_Lectura_0_20","Prueba_Matemática_0_20"]].mean()
print("8) Lectura masculino:", promedios.loc["M","Prueba_Lectura_0_20"])
print("9) Matemática masculino:", promedios.loc["M","Prueba_Matemática_0_20"])
print("10) Lectura femenino:", promedios.loc["F","Prueba_Lectura_0_20"])
print("11) Matemática femenino:", promedios.loc["F","Prueba_Matemática_0_20"])

# ===============================
# 12-15. Aprobados y desaprobados Matemática
# ===============================
df["Resultado_Mat"] = np.where(df["Prueba_Matemática_0_20"]>=11,"Aprobado","Desaprobado")
conteo_mat = df["Resultado_Mat"].value_counts()
porc_mat = df["Resultado_Mat"].value_counts(normalize=True)*100
print("12) Nº Aprobados:", conteo_mat.get("Aprobado",0))
print("13) % Aprobados:", porc_mat.get("Aprobado",0))
print("14) Nº Desaprobados:", conteo_mat.get("Desaprobado",0))
print("15) % Desaprobados:", porc_mat.get("Desaprobado",0))

# ===============================
# 16-18. Mediana de edad por región
# ===============================
for region in ["Costa","Sierra","Selva"]:
    mediana = df.loc[df["Región"]==region,"Edad"].median()
    print(f"Mediana de Edad en {region}: {mediana}")

# ===============================
# 19. Correlación entre Lectura y Matemática
# ===============================
corr = df["Prueba_Lectura_0_20"].corr(df["Prueba_Matemática_0_20"])
print("19) Correlación Lectura vs Matemática:", corr)

# ===============================
# 20. Brecha F-M en conocimiento SDSR
# ===============================
filtro_sdsr = df[df["Conoce_DSR_(0=no,1=sí)"]==1]
tasas = filtro_sdsr.groupby("Sexo")["Conoce_DSR_(0=no,1=sí)"].mean()*100
brecha = tasas.get("F",0) - tasas.get("M",0)
print("20) Brecha conocimiento SDSR (F-M) en p.p.:", brecha)

Columnas del dataset: ['ID_Participante', 'Región', 'Provincia', 'Distrito', 'Sexo', 'Edad', 'Escuela', 'Grado', 'Matriculado', 'Asistencia_%', 'Prueba_Lectura_0_20', 'Prueba_Matemática_0_20', 'Conoce_DSR_(0=no,1=sí)', 'Fecha_Registro', 'Proyecto', 'Condición_Vulnerable']
1) Filas con IDs duplicados: 10
2) Registros con Asistencia_% en blanco: 6
3) Promedio Asistencia_% (Matriculados=Sí, Región=Sierra): 83.39999999999998
4) Recuento masculinos: 96
5) % masculinos: 53.333333333333336
6) Recuento femeninos: 84
7) % femeninos: 46.666666666666664
8) Lectura masculino: 11.479166666666666
9) Matemática masculino: 10.885416666666666
10) Lectura femenino: 12.011904761904763
11) Matemática femenino: 10.761904761904763
12) Nº Aprobados: 98
13) % Aprobados: 54.44444444444444
14) Nº Desaprobados: 82
15) % Desaprobados: 45.55555555555556
Mediana de Edad en Costa: 15.0
Mediana de Edad en Sierra: 15.0
Mediana de Edad en Selva: 15.0
19) Correlación Lectura vs Matemática: -0.07749265981766067
20) Brech

In [4]:
# 20. Brecha F-M en conocimiento SDSR
# ===============================

tasas = df.groupby("Sexo")["Conoce_DSR_(0=no,1=sí)"].mean() * 100  # % que responden 1 en cada sexo
brecha = tasas.get("F", 0) - tasas.get("M", 0)

print("20) Brecha conocimiento SDSR (F-M) en p.p.:", round(brecha, 2))


20) Brecha conocimiento SDSR (F-M) en p.p.: 3.57
